In [7]:
import pandas as pd

# Charger le CSV
df = pd.read_csv("USD.txt", sep=",")

# On garde l'essentiel pour la première version
df = df[['match_id','player1','player2','point_no','points_victor']]
# Par exemple, on ne prend que les 50 premiers matchs
df = df[df['match_id'].isin(df['match_id'].unique()[:50])]


# Vérification
print(df.head())


C:\Users\pc\AppData\Local\Temp\ipykernel_26192\1229551701.py:4: DtypeWarning: Columns (11,12) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("USD.txt", sep=",")


           match_id  player1  player2  point_no  points_victor
0  2013-usopen-1101      NaN      NaN         1              2
1  2013-usopen-1101      NaN      NaN         2              1
2  2013-usopen-1101      NaN      NaN         3              1
3  2013-usopen-1101      NaN      NaN         4              1
4  2013-usopen-1101      NaN      NaN         5              1


In [8]:
import numpy as np

def create_point_sequences(df, past_len=10, future_len=5):
    X, Y = [], []
    
    # Parcours match par match
    for match_id, match_data in df.groupby('match_id'):
        points = match_data['points_victor'].values  # 1 = joueur1, 0 = joueur2
        # Créer les fenêtres
        for i in range(len(points) - past_len - future_len + 1):
            X.append(points[i:i+past_len])
            Y.append(points[i+past_len:i+past_len+future_len])
    
    return np.array(X), np.array(Y)

X, Y = create_point_sequences(df)
print(X.shape, Y.shape)


(10276, 10) (10276, 5)


In [9]:
import torch
from torch.utils.data import TensorDataset, DataLoader

X_tensor = torch.tensor(X, dtype=torch.float32).unsqueeze(-1)  # shape (batch, seq, 1)
Y_tensor = torch.tensor(Y, dtype=torch.float32).unsqueeze(-1)  # shape (batch, seq, 1)

dataset = TensorDataset(X_tensor, Y_tensor)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)


In [10]:
import torch.nn as nn

class Encoder(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=64, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)

    def forward(self, x):
        outputs, (h, c) = self.lstm(x)
        return h, c

class Decoder(nn.Module):
    def __init__(self, output_dim=1, hidden_dim=64, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(output_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x, hidden):
        out, hidden = self.lstm(x, hidden)
        out = self.fc(out)
        return out, hidden

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, future_len=5):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.future_len = future_len

    def forward(self, src):
        batch_size = src.size(0)
        h, c = self.encoder(src)

        decoder_input = torch.zeros(batch_size, 1, 1).to(src.device)
        outputs = []
        hidden = (h, c)
        
        for _ in range(self.future_len):
            out, hidden = self.decoder(decoder_input, hidden)
            outputs.append(out)
            decoder_input = out  # auto-regressive

        return torch.cat(outputs, dim=1)


In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder = Encoder()
decoder = Decoder()
model = Seq2Seq(encoder, decoder, future_len=Y.shape[1]).to(device)

criterion = nn.BCEWithLogitsLoss()  # car point = 0 ou 1
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 5  # pour tester
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for xb, yb in dataloader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(dataloader):.4f}")


Epoch 1, Loss: -3.8233
Epoch 2, Loss: -10.2065
Epoch 3, Loss: -15.6504
Epoch 4, Loss: -20.9801
Epoch 5, Loss: -26.2070


In [12]:
model.eval()
with torch.no_grad():
    sample = X_tensor[:1].to(device)  # prendre une séquence
    pred = torch.sigmoid(model(sample)).cpu().numpy()
    print("Prédiction des prochains points :", pred.squeeze())


Prédiction des prochains points : [1. 1. 1. 1. 1.]
